# MLP Baseline — Polymarket Resolution Prediction

Feedforward нейросеть (PyTorch) на 5 базовых фичах рынка. Сравнение с LightGBM.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'

import torch
torch.set_num_threads(1)  # избегаем OMP конфликт с LightGBM на macOS
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA = Path('../../data/processed')
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')

## Данные и feature engineering

In [ ]:
# Загрузим реальные данные: resolution outcomes (322K resolved markets)
outcomes = pd.read_parquet(DATA / 'resolution_outcomes.parquet')
resolved = outcomes[outcomes['outcome'].isin([0, 1])].copy()
print(f'Resolved markets: {len(resolved)}')
print(f'Outcome distribution: {resolved.outcome.value_counts().to_dict()}')

In [ ]:
# Feature engineering — используем только фичи доступные ДО resolution
# lastTradePrice — leaky (цена при закрытии ≈ outcome), исключаем
resolved['negRisk'] = resolved['negRisk'].astype(int)
resolved['market_age_days'] = (
    pd.to_datetime(resolved['closedTime']) - pd.to_datetime(resolved['createdAt'])
).dt.total_seconds() / 86400
resolved['log_volume'] = np.log1p(resolved['volumeNum'].fillna(0))
resolved['log_liquidity'] = np.log1p(resolved['liquidityClob'].fillna(0))

# Честные фичи: volume, liquidity, spread, negRisk, market age
feature_cols = ['log_volume', 'log_liquidity', 'spread', 'negRisk', 'market_age_days']
df = resolved.dropna(subset=feature_cols + ['outcome', 'closedTime']).copy()

print(f'Dataset: {len(df)} rows, {len(feature_cols)} features')
print(f'⚠️ Без lastTradePrice (leaky) — честная оценка')
df[feature_cols + ['outcome']].describe()

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

class PolymarketDataset(Dataset):
    """PyTorch Dataset для данных Polymarket."""
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Temporal split (70/15/15)
df = df.sort_values('closedTime')
n = len(df)
train_df = df.iloc[:int(n*0.7)]
val_df = df.iloc[int(n*0.7):int(n*0.85)]
test_df = df.iloc[int(n*0.85):]

# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols])
X_val = scaler.transform(val_df[feature_cols])
X_test = scaler.transform(test_df[feature_cols])

y_train = train_df['outcome'].values
y_val = val_df['outcome'].values
y_test = test_df['outcome'].values

# DataLoaders
train_ds = PolymarketDataset(X_train, y_train)
val_ds = PolymarketDataset(X_val, y_val)
test_ds = PolymarketDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512)
test_loader = DataLoader(test_ds, batch_size=512)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Batches per epoch: {len(train_loader)}')

## MLP: архитектура и обучение

In [ ]:
class MarketMLP(nn.Module):
    """Простая нейросеть для предсказания исхода рынка."""
    def __init__(self, n_features, hidden_dims=[64, 32]):
        super().__init__()
        layers = []
        prev_dim = n_features
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.ReLU(),
                nn.Dropout(0.2),
            ])
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))  # output logit
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x).squeeze(-1)   # (batch,)

# На CPU для стабильности (MPS + LightGBM конфликтуют в headless mode)
model = MarketMLP(n_features=len(feature_cols))
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
from sklearn.metrics import roc_auc_score

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()  # sigmoid + BCE в одном (числено стабильнее)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    total_loss = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = model(X)
        total_loss += criterion(logits, y).item() * len(y)
        all_preds.append(torch.sigmoid(logits).cpu())
        all_targets.append(y.cpu())
    preds = torch.cat(all_preds).numpy()
    targets = torch.cat(all_targets).numpy()
    auc = roc_auc_score(targets, preds) if len(np.unique(targets)) > 1 else 0.5
    return total_loss / len(loader.dataset), auc

# Training loop
EPOCHS = 30
history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
best_auc = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        
        optimizer.zero_grad()           # 1. обнулить градиенты
        logits = model(X_batch)         # 2. forward pass
        loss = criterion(logits, y_batch)  # 3. считаем loss
        loss.backward()                 # 4. backward pass (градиенты)
        optimizer.step()                # 5. обновить веса
        
        epoch_loss += loss.item() * len(y_batch)
    
    train_loss = epoch_loss / len(train_ds)
    val_loss, val_auc = evaluate(model, val_loader)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), '../../data/models/mlp_baseline.pt')
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
              f'Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | '
              f'Val AUC: {val_auc:.4f} {"★" if val_auc >= best_auc else ""}')

In [ ]:
# Визуализация обучения
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss')
ax1.legend(); ax1.set_title('Loss Curves')

ax2.plot(history['val_auc'], 'g-o', markersize=3)
ax2.axhline(y=best_auc, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_auc:.4f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('AUC')
ax2.legend(); ax2.set_title('Validation AUC')

plt.tight_layout(); plt.show()

## Сравнение с LightGBM

In [ ]:
import lightgbm as lgb

# LightGBM на тех же данных / фичах
lgb_model = lgb.LGBMClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    random_state=SEED, verbose=-1
)
lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)])

lgb_val_auc = roc_auc_score(y_val, lgb_model.predict_proba(X_val)[:, 1])
lgb_test_auc = roc_auc_score(y_test, lgb_model.predict_proba(X_test)[:, 1])

# MLP test AUC
model.load_state_dict(torch.load('../../data/models/mlp_baseline.pt', weights_only=True))
mlp_test_loss, mlp_test_auc = evaluate(model, test_loader)

print(f'\n{"="*45}')
print(f'{"Model":<18} {"Val AUC":>10} {"Test AUC":>10}')
print(f'{"-"*45}')
print(f'{"MLP (PyTorch)":<18} {best_auc:>10.4f} {mlp_test_auc:>10.4f}')
print(f'{"LightGBM":<18} {lgb_val_auc:>10.4f} {lgb_test_auc:>10.4f}')
print(f'{"HTR (prod, 14ft)":<18} {"—":>10} {"0.9591":>10}')
print(f'{"="*45}')
print(f'\nМы используем {len(feature_cols)} базовых фич (HTR — 14 rich mid-life).')
print('Вывод: фичи > модель. DL раскроется на NLP и time series.')